# Study 293 -- MVRV-Ratio
## For the quants: contrarian regression, price-momentum horse race, per-band returns, timing vs buy-and-hold

*Part of [Open-Alpha-Lab](../../../README.md). See the [desk methodology](../../../METHODOLOGY.md).*


## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"

from mvrv_ratio import data, strategy as st

CACHE_PATH = data.BTC_CACHE
HAVE_REAL = os.path.exists(CACHE_PATH)

if HAVE_REAL:
    df = data.joined_real(fetch=False, cache_path=CACHE_PATH)
    reg = st.predictive_regression(df)
    pos = st.timing_signal(df, high=3.5, low=1.0)
    bt = st.backtest_timing(df, pos, cost_bps=30.0)
    print(f"Real tape: {len(df)} aligned months  {df.index[0].date()} -> {df.index[-1].date()}")
else:
    df = reg = pos = bt = None
    print("No real BTC cache -- frozen headline numbers from R dict will be used")


Real tape: 141 aligned months  2014-09-30 -> 2026-05-31


In [2]:

# Frozen headline numbers (mirror of docs/results.md, as-of 2026-06-17)
R = {'n_months': 140, 'n_mvrv': 149, 'reg_slope': -0.0067, 'reg_t': -0.15, 'reg_r2': 0.0, 'horse_mvrv_t': -0.7, 'horse_price_t': 1.88, 'band_over_mo': -6.7, 'band_over_n': 4, 'band_over_hit': 0.25, 'band_neutral_mo': 6.39, 'band_neutral_n': 111, 'band_under_mo': 4.97, 'band_under_n': 25, 'tim_share': 0.972, 'tim_turnover': 0.029, 'timing_ann': 71.4, 'timing_t': 2.89, 'timing_sr': 1.03, 'bh_ann': 69.2, 'bh_t': 2.74, 'bh_sr': 0.97, 'excess_ann': 2.2, 'excess_t': 0.59, 'high30_ann': 60.1, 'high40_ann': 71.5, 'syn_slope': -0.474, 'syn_t': -9.6, 'syn_null_t': 0.52}


## Positive control: the engine detects a planted contrarian MVRV->price link

In [3]:
# Synthetic positive control: beta=0.50 plants last-month MVRV stretch NEGATIVELY
# into this-month price return (the contrarian mechanism). The regression should
# recover a strongly negative slope.
df_syn, truth = data.synthetic_series(beta=0.50, seed=293)
reg_syn = st.predictive_regression(df_syn)
print(f"Positive control (beta=0.50): slope = {reg_syn['slope_mvrv']:+.3f}  HAC t = {reg_syn['t_mvrv']:+.2f}  n = {reg_syn['n']}")

# Null control: beta=0.0 -> MVRV is an independent mean-reverting series
df_null, _ = data.synthetic_series(beta=0.0, seed=293)
reg_null = st.predictive_regression(df_null)
print(f"Null control (beta=0.00):     slope = {reg_null['slope_mvrv']:+.3f}  HAC t = {reg_null['t_mvrv']:+.2f}")
print("\n-> Engine reads strongly negative on a planted contrarian link, ~zero on null. It is truthful.")


Positive control (beta=0.50): slope = -0.474  HAC t = -9.60  n = 143
Null control (beta=0.00):     slope = +0.026  HAC t = +0.52

-> Engine reads strongly negative on a planted contrarian link, ~zero on null. It is truthful.


## Real tape: predictive regression of next-month return on MVRV stretch

In [4]:
if HAVE_REAL:
    r = st.predictive_regression(df)
    print(f"r(t+1) = a + b*MVRV_stretch(t)")
    print(f"  slope={r['slope_mvrv']:+.4f}  HAC t={r['t_mvrv']:+.2f}  R^2={r['r2']:.4f}  n={r['n']}")
else:
    print(f"  slope={R['reg_slope']:+.4f}  HAC t={R['reg_t']:+.2f}  R^2={R['reg_r2']:.4f}  n={R['n_months']}")
print("\nThe slope is the right (negative) sign for the contrarian story but does")
print("NOT clear |t| >= 2. MVRV stretch is not a leading indicator of BTC returns.")


r(t+1) = a + b*MVRV_stretch(t)
  slope=-0.0067  HAC t=-0.15  R^2=0.0002  n=140

The slope is the right (negative) sign for the contrarian story but does
NOT clear |t| >= 2. MVRV stretch is not a leading indicator of BTC returns.


## Horse race: does MVRV add anything beyond BTC's own momentum?

In [5]:
if HAVE_REAL:
    rc = st.predictive_regression(df, add_price_control=True)
    print("r(t+1) = a + b*MVRV_stretch(t) + c*price_momentum(t)")
    print(f"  MVRV slope b: HAC t = {rc['t_mvrv']:+.2f}")
    print(f"  price-mom slope c: HAC t = {rc['t_price']:+.2f}")
else:
    print(f"  MVRV slope: HAC t = {R['horse_mvrv_t']:+.2f}")
    print(f"  price-mom slope: HAC t = {R['horse_price_t']:+.2f}")
print("\nWith price momentum in the regression, MVRV's t-stat stays ~0. Whatever")
print("little contrarian whiff it carries is not incremental to the price trend.")


r(t+1) = a + b*MVRV_stretch(t) + c*price_momentum(t)
  MVRV slope b: HAC t = -0.70
  price-mom slope c: HAC t = +1.88

With price momentum in the regression, MVRV's t-stat stays ~0. Whatever
little contrarian whiff it carries is not incremental to the price trend.


## Per-band forward returns: the 'sell' band rests on n=4

In [6]:
if HAVE_REAL:
    tab = st.state_forward_stats(df, high=3.5, low=1.0)
    print(tab.to_string())
else:
    print("over-heated  : %+.2f%%/mo  hit=%.2f  n=%d" % (R['band_over_mo'], R['band_over_hit'], R['band_over_n']))
    print("neutral      : %+.2f%%/mo            n=%d" % (R['band_neutral_mo'], R['band_neutral_n']))
    print("under-valued : %+.2f%%/mo            n=%d" % (R['band_under_mo'], R['band_under_n']))
print("\nThe over-heated band's negative forward return is real but rests on FOUR")
print("months (2017 + 2021 tops). The under-valued 'buy' band does not beat neutral.")
print("With n=4 the 'sell the top' edge is anecdote, not a testable signal.")


                  mean    n       hit
band                                 
over-heated  -0.067011    4  0.250000
neutral       0.063932  111  0.567568
under-valued  0.049725   25  0.560000

The over-heated band's negative forward return is real but rests on FOUR
months (2017 + 2021 tops). The under-valued 'buy' band does not beat neutral.
With n=4 the 'sell the top' edge is anecdote, not a testable signal.


## Contrarian timing rule vs buy-and-hold (net of costs)

In [7]:
if HAVE_REAL:
    s_net = st.summarize(bt["net"]); s_gross = st.summarize(bt["gross"]); s_bh = st.summarize(bt["bh"])
    print(f"Time in market: {st.time_in_market(pos):.1%}   avg turnover: {st.turnover(pos):.3f}/mo")
    print(f"GROSS timing: {s_gross['mean']*1200:+.1f}%/yr  SR={s_gross['sharpe']*12**0.5:+.2f}  HAC t={s_gross['tstat']:+.2f}")
    print(f"NET   timing: {s_net['mean']*1200:+.1f}%/yr  SR={s_net['sharpe']*12**0.5:+.2f}  HAC t={s_net['tstat']:+.2f}")
    print(f"BUY-HOLD:     {s_bh['mean']*1200:+.1f}%/yr  SR={s_bh['sharpe']*12**0.5:+.2f}  HAC t={s_bh['tstat']:+.2f}")
    excess = (bt['net'] - bt['bh'])
    se = st.summarize(excess)
    print(f"\nTiming minus buy-hold: {se['mean']*1200:+.1f}%/yr  HAC t={se['tstat']:+.2f}")
else:
    print(f"Time in market: {R['tim_share']:.1%}   avg turnover: {R['tim_turnover']:.3f}/mo")
    print(f"NET   timing: {R['timing_ann']:+.1f}%/yr  SR={R['timing_sr']:+.2f}  HAC t={R['timing_t']:+.2f}")
    print(f"BUY-HOLD:     {R['bh_ann']:+.1f}%/yr  SR={R['bh_sr']:+.2f}  HAC t={R['bh_t']:+.2f}")
    print(f"\nTiming minus buy-hold: {R['excess_ann']:+.1f}%/yr  HAC t={R['excess_t']:+.2f}")
print("\nThe rule is 97% buy-and-hold. Its +2%/yr edge over holding is NOT significant")
print("(HAC t ~ 0.6) and comes entirely from sidestepping a handful of post-top months.")


Time in market: 97.2%   avg turnover: 0.029/mo


GROSS timing: +71.5%/yr  SR=+1.03  HAC t=+2.90
NET   timing: +71.4%/yr  SR=+1.03  HAC t=+2.89
BUY-HOLD:     +69.2%/yr  SR=+0.97  HAC t=+2.74

Timing minus buy-hold: +2.2%/yr  HAC t=+0.59

The rule is 97% buy-and-hold. Its +2%/yr edge over holding is NOT significant
(HAC t ~ 0.6) and comes entirely from sidestepping a handful of post-top months.


## Band-threshold sensitivity: a knob, not a signal

In [8]:
if HAVE_REAL:
    for h in (3.0, 3.5, 4.0):
        p = st.timing_signal(df, high=h, low=1.0)
        b = st.backtest_timing(df, p, cost_bps=30.0)
        s = st.summarize(b["net"])
        print(f"high={h:.1f}: net {s['mean']*1200:+.1f}%/yr  SR={s['sharpe']*12**0.5:+.2f}  long {st.time_in_market(p):.0%}")
    s_bh = st.summarize(bt["bh"])
    print(f"buy-hold : {s_bh['mean']*1200:+.1f}%/yr  SR={s_bh['sharpe']*12**0.5:+.2f}")
else:
    print(f"high=3.0: net {R['high30_ann']:+.1f}%/yr  (trails buy-hold)")
    print(f"high=3.5: net {R['timing_ann']:+.1f}%/yr  (the cherry-picked sweet spot)")
    print(f"high=4.0: net {R['high40_ann']:+.1f}%/yr  (~ buy-hold, ~99% in market)")
    print(f"buy-hold: {R['bh_ann']:+.1f}%/yr")
print("\nTighten the band and it trails buy-hold; loosen it and it converges to buy-hold.")
print("The 'edge' is a threshold-tuning artefact, the signature of a mirage.")


high=3.0: net +60.1%/yr  SR=+0.91  long 94%
high=3.5: net +71.4%/yr  SR=+1.03  long 97%


high=4.0: net +71.5%/yr  SR=+1.01  long 99%
buy-hold : +69.2%/yr  SR=+0.97

Tighten the band and it trails buy-hold; loosen it and it converges to buy-hold.
The 'edge' is a threshold-tuning artefact, the signature of a mirage.


## Cost & lag honesty

In [9]:
print("Honesty checklist:")
print(" - Execution lag: MVRV known at month-end t, position held for month t+1 (1-month lag).")
print(" - Costs: 30 bps one-way charged on every flip (|delta position|) x NAV. Long/flat -> no borrow.")
print(" - Returns: PRICE-ONLY (BTC pays no yield); same basis for timing and buy-hold.")
print(" - Single-survivor: BTC is the one crypto that 1000x'd and MVRV is DERIVED from its")
print("   own price path (realized cap = slow average of past prices). The contrarian bands")
print("   are fitted to ~four cycle turns. NAMED on the Signal axis.")


Honesty checklist:
 - Execution lag: MVRV known at month-end t, position held for month t+1 (1-month lag).
 - Costs: 30 bps one-way charged on every flip (|delta position|) x NAV. Long/flat -> no borrow.
 - Returns: PRICE-ONLY (BTC pays no yield); same basis for timing and buy-hold.
 - Single-survivor: BTC is the one crypto that 1000x'd and MVRV is DERIVED from its
   own price path (realized cap = slow average of past prices). The contrarian bands
   are fitted to ~four cycle turns. NAMED on the Signal axis.


## Verdict

In [10]:
print("=== Study 293 -- MVRV-Ratio ===")
print()
print(f"Signal: NONE")
print(f"  MVRV stretch does not predict next-month BTC returns: HAC t = {R['reg_t']:+.2f} (R^2 ~ 0).")
print(f"  In a horse race vs price momentum the MVRV slope is t = {R['horse_mvrv_t']:+.2f}. The only")
print(f"  suggestive number -- the over-heated band's {R['band_over_mo']:+.1f}%/mo -- rests on n = {R['band_over_n']} months.")
print()
print(f"Tradability: MIRAGE")
print(f"  The contrarian 'cash when over-heated' rule is {R['tim_share']:.0%} buy-and-hold; its")
print(f"  {R['excess_ann']:+.1f}%/yr edge over holding is insignificant (HAC t = {R['excess_t']:+.2f}) and flips")
print(f"  sign with the band threshold. Any CAGR is just long exposure to a 1000x survivor.")
print()
print("Single-survivor: NAMED -- BTC is the surviving moonshot; MVRV is derived from its price.")
print()
print("Bottom line: None/Mirage -- a beloved on-chain gauge that is hindsight")
print("band-fitting on a single survivor, with zero incremental predictive content.")


=== Study 293 -- MVRV-Ratio ===

Signal: NONE
  MVRV stretch does not predict next-month BTC returns: HAC t = -0.15 (R^2 ~ 0).
  In a horse race vs price momentum the MVRV slope is t = -0.70. The only
  suggestive number -- the over-heated band's -6.7%/mo -- rests on n = 4 months.

Tradability: MIRAGE
  The contrarian 'cash when over-heated' rule is 97% buy-and-hold; its
  +2.2%/yr edge over holding is insignificant (HAC t = +0.59) and flips
  sign with the band threshold. Any CAGR is just long exposure to a 1000x survivor.

Single-survivor: NAMED -- BTC is the surviving moonshot; MVRV is derived from its price.

Bottom line: None/Mirage -- a beloved on-chain gauge that is hindsight
band-fitting on a single survivor, with zero incremental predictive content.
